In [2]:
import os
import sys
import datetime as dt
import cmocean as cmo

import cartopy.crs as ccrs
import easygems.healpix as egh
import intake
import matplotlib.pyplot as plt
import numpy as np

# import healpy as hp
import xarray as xr

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
from utils import haversine, hp_mods, get_nn_lon_lat_index, hp_to_latlon
    
# Filter out annoying warning.
import warnings


warnings.filterwarnings(
    "ignore",
    message=".*The return type of `Dataset.dims` will be changed.*",
    category=FutureWarning,
)

In [3]:
latmin=8.;latmax=29.;lonmin=69.;lonmax=89.
projection = ccrs.PlateCarree()
fig, ax = plt.subplots(
    subplot_kw={"projection": projection},
    layout="constrained",
)
ax.plot([lonmin,lonmax],[latmin,latmin],"k")
ax.plot([lonmin,lonmax],[latmax,latmax],"k")
ax.plot([lonmin,lonmin],[latmin,latmax],"k")
ax.plot([lonmax,lonmax],[latmin,latmax],"k")
ax.coastlines()
ax.gridlines(draw_labels=True)
dx=4
ax.set_extent([lonmin-dx,lonmax+dx,latmin-dx,latmax+dx])

In [4]:
def get_rainfall(sim, zoom, region=None):
    url = "https://digital-earths-global-hackathon.github.io/catalog/catalog.yaml"
    cat = intake.open_catalog(url)["UK"]
    sim_cat = cat[sim]
    if "hk26" in sim:
        ds = sim_cat(zoom=zoom, time="PT1H").to_dask().pipe(hp_mods).pr
    else:
        if "icon" in sim or "nicam" in sim:
            ds = sim_cat(zoom=zoom).to_dask().pipe(egh.attach_coords).pr
        elif "IMERG" in sim:
            zoom=9
            ds = sim_cat(zoom=zoom).to_dask().pipe(egh.attach_coords).precipitation
        elif "arp" in sim:
            zoom=8
            ds = sim_cat(zoom=zoom).to_dask().pipe(egh.attach_coords).pr
        else:
            ds = sim_cat(zoom=zoom, time="PT1H").to_dask().pipe(egh.attach_coords).pr
    return hp_to_latlon(ds,zoom)

In [12]:
sims = [
    # "um_glm_n1280_GAL9_v2_hk26",
    # "um_glm_n2560_CoMA9_hk26",
    # "um_glm_n1280_CoMA9_hk26",
    "um_glm_n2560_RAL3p3_tuned_hk26",
    # "ifs_tco3999-ng5_rcbmf_cf",
    "icon_d3hp003",
    "casesm2_10km_nocumulus",
    "nicam_gl11",
    # "arp-gem-2p6km",
    "IR_IMERG",
]
labels = ["UM RAL3", "ICON", "CAS-ESM", "NICAM"]
zoom = 3

for sim_ix, sim in enumerate(sims):
    print(sim)
    get_rainfall(sim, zoom).sel(
        longitude=slice(lonmin, lonmax), latitude=slice(latmin, latmax)
    ).resample(time="5D").mean().mean(dim=("longitude", "latitude")).plot(
        label=labels[sim_ix]
    )

plt.legend()
plt.title("Indian Monsoon Region")

In [ ]:
domains = {
    "SAm": {"lonmin": -80, "lonmax": -30, "latmin": -40, "latmax": 10},
    "SAf": {"lonmin": 10, "lonmax": 55, "latmin": -40, "latmax": 0},
    "WAf": {"lonmin": -30, "lonmax": 30, "latmin": -5, "latmax": 25},
    "SAs": {"lonmin": 60, "lonmax": 100, "latmin": 0, "latmax": 35},
    "Ind": {"lonmin": 69, "lonmax": 89, "latmin": 8, "latmax": 29},
    "EAs": {"lonmin": 90, "lonmax": 140, "latmin": 0, "latmax": 50},
    "Aus": {"lonmin": 110, "lonmax": 160, "latmin": -30, "latmax": 0},
}

sims = [
    # "um_glm_n1280_GAL9_v2_hk26",
    # "um_glm_n2560_CoMA9_hk26",
    # "um_glm_n1280_CoMA9_hk26",
    "um_glm_n2560_RAL3p3_tuned_hk26",
    # "ifs_tco3999-ng5_rcbmf_cf",
    # "icon_d3hp003",
    # "casesm2_10km_nocumulus",
    # "nicam_gl11",
    # "arp-gem-2p6km",
    # "IR_IMERG",
]
zoom = 3
for region in domains:
    fig,ax=plt.subplots
    lonmin = domains[region]["lonmin"]
    lonmax = domains[region]["lonmax"]
    latmin = domains[region]["latmin"]
    latmax = domains[region]["latmax"]
    for sim in sims:
        print(sim)
        outdir = f"precip/{sim}_{region}_zoom_{zoom}.nc"
        pr = (
            get_rainfall(sim, zoom)
            .sel(longitude=slice(lonmin, lonmax), latitude=slice(latmin, latmax))
            .mean(dim=("longitude", "latitude"))
        )
        pr.to_netcdf(outdir)